# Notebook Overview — Build Dataset (Optional)

## Purpose

This notebook provides an **optional, reproducible pipeline** for building a raw image dataset from **one selected source at a time**.

It retrieves candidate images, applies validation and deduplication, assigns standardized filenames, and generates raw metadata records for accepted images.

This step is **optional** because prebuilt datasets from all six sources are provided as ZIP archives and can be extracted and used directly in subsequent notebooks.

The notebook is useful for:

* Demonstrating the full dataset construction pipeline
* Rebuilding or extending individual dataset sources
* Experimenting with filtering and deduplication strategies

---

## Inputs

Supported dataset sources:

* ImageNet_1K_256
* MS_COCO_2017
* DiffusionDB
* SDXL_Generated_10K
* MidJourney
* OpenImages *(dataset download intentionally disabled due to size constraints)*

💡 Alternatively, **prebuilt datasets are provided as ZIP archives** and can be extracted for direct use in subsequent notebooks, allowing this notebook to be skipped.

Only one dataset source is processed per run.

Target-specific dataset modules (used for dataset loading and candidate record retrieval):

* src/datasets/imagenet_target.py
* src/datasets/coco_target.py
* src/datasets/diffusiondb_target.py
* src/datasets/sdxl_target.py
* src/datasets/midjourney_target.py
* src/datasets/openimages_target.py

Dataset categories:

| Target Name | Dataset              | Class Label | Dataset Code |
|------------|----------------------|-------------|--------------|
| imagenet   | ImageNet_1K_256      | rl          | imgn         |
| coco       | MS_COCO_2017         | rl          | coco         |
| openimages | OpenImages           | rl          | open         |
| diffusiondb| DiffusionDB          | ai          | diff         |
| sdxl       | SDXL_Generated_10K   | ai          | sdxl         |
| midjourney | MidJourney           | ai          | midj         |

---

## Execution Model

* Dataset construction is performed **one source at a time**
* Accepted images must satisfy:

  * width ≥ 256
  * height ≥ 256

* Duplicate detection uses **SHA-256 hashes**
* Deduplication occurs at two levels:

  * per-source
  * global (cross-source)

* No preprocessing or dataset splitting is performed in this notebook
* A reset mechanism enables clean rebuilding of any dataset source
* Execution is deterministic and repeatable

---

## Outputs

For the selected dataset source:

**Images**  
`data/raw/<SOURCE_DATASET>/images/`  
Accepted images saved without preprocessing, with pixel content preserved, in a consistent PNG format.

**Raw Metadata CSV**  
`data/metadata/original/<dataset_code>_raw_metadata.csv`  
Records one entry per accepted image, capturing identifiers, source information, and file paths.

**Per-Source Hash File**  
`data/metadata/hashes/<dataset_code>_source_hashes.json`  
Tracks hashes for the current dataset source to prevent duplicate images within the source.

**Global Hash File**  
`data/metadata/hashes/global_hashes.json`  
Tracks hashes across all dataset sources to prevent cross-source duplication.

**Expected Sizes**

* ~3000 images per dataset source  
* ~18,000 images total across all datasets

---
---

### 🔷 Step 1 — Environment Setup and Configuration

⚙️ This step initializes the notebook environment and loads project configuration.

- Define project paths (`BASE_DIR`, `src`, config file)
- Clone the GitHub repository if not already present
- Verify required directory structure and configuration file
- Add `src` directory to the Python path
- Dynamically import `project_config.py`
- Extract required configuration parameters for this notebook
- Create required data and metadata directories (if missing)
- Optionally display configuration details when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 1: Environment Setup and Configuration
# ============================================

# ============================================
# USER CONFIGURATION — EDIT THIS SECTION ONLY
# ============================================

VERBOSE = True   # User toggle (True or False)

# ============================================
# END USER CONFIGURATION
# ============================================

import os
import sys
import importlib.util

# -------------------------------------------------
# Define project locations
# -------------------------------------------------
BASE_DIR = "/content/dip-ai-image-detection"
PROJECT_SRC_DIR = f"{BASE_DIR}/src"
CONFIG_FILE = f"{PROJECT_SRC_DIR}/project_config.py"

print("Initializing environment...")

# -------------------------------------------------
# Clone repo if not already present
# -------------------------------------------------
if not os.path.isdir(BASE_DIR):
    print("Cloning repository...")
    !git clone -q https://github.com/pgailinas/dip-ai-image-detection.git
else:
    print("Repository already exists. Skipping clone.")

# -------------------------------------------------
# Verify structure (always critical)
# -------------------------------------------------
if not os.path.isdir(PROJECT_SRC_DIR):
    raise FileNotFoundError(f"Missing directory: {PROJECT_SRC_DIR}")

if not os.path.isfile(CONFIG_FILE):
    raise FileNotFoundError(f"Missing file: {CONFIG_FILE}")

# -------------------------------------------------
# Add src to path
# -------------------------------------------------
sys.path.insert(0, PROJECT_SRC_DIR)

# -------------------------------------------------
# Import config
# -------------------------------------------------
spec = importlib.util.spec_from_file_location("project_config", CONFIG_FILE)
cfg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cfg)

# -------------------------------------------------
# Pull only what this notebook actually uses
# -------------------------------------------------
TARGET_COUNT          = cfg.TARGET_COUNT
BATCH_SIZE            = cfg.BATCH_SIZE
RANDOM_SEED           = cfg.RANDOM_SEED
RAW_DATA_DIR          = cfg.RAW_DATA_DIR
ORIGINAL_METADATA_DIR = cfg.ORIGINAL_METADATA_DIR
HASH_METADATA_DIR     = cfg.HASH_METADATA_DIR

# -------------------------------------------------
# Ensure required directories exist
# -------------------------------------------------
os.makedirs(cfg.DATA_DIR, exist_ok=True)
os.makedirs(cfg.RAW_DATA_DIR, exist_ok=True)
os.makedirs(cfg.METADATA_DIR, exist_ok=True)
os.makedirs(cfg.ORIGINAL_METADATA_DIR, exist_ok=True)
os.makedirs(cfg.HASH_METADATA_DIR, exist_ok=True)

# -------------------------------------------------
# Optional display block
# -------------------------------------------------
print("\nEnvironment setup complete.\n")

if VERBOSE:
    print(f"BASE_DIR          : {cfg.BASE_DIR}")
    print(f"DATA_DIR          : {cfg.DATA_DIR}")
    print(f"RAW_DATA_DIR      : {cfg.RAW_DATA_DIR}")
    print(f"METADATA_DIR      : {cfg.METADATA_DIR}")
    print(f"ORIGINAL_METADATA : {cfg.ORIGINAL_METADATA_DIR}")
    print(f"HASH_METADATA     : {cfg.HASH_METADATA_DIR}")
    print(f"TARGET_COUNT      : {TARGET_COUNT}")
    print(f"BATCH_SIZE        : {BATCH_SIZE}")
    print("\nOutputs will be written to local project directories.")



Initializing environment...
Cloning repository...

Environment setup complete.

BASE_DIR          : /content/dip-ai-image-detection
DATA_DIR          : /content/dip-ai-image-detection/data
RAW_DATA_DIR      : /content/dip-ai-image-detection/data/raw
METADATA_DIR      : /content/dip-ai-image-detection/metadata
ORIGINAL_METADATA : /content/dip-ai-image-detection/metadata/original
HASH_METADATA     : /content/dip-ai-image-detection/metadata/hashes
TARGET_COUNT      : 3000
BATCH_SIZE        : 200

Outputs will be written to local project directories.


### 🔷 Step 2 — Download Mode and Target Selection

- Display a user prompt for selecting the Notebook 01 execution mode
- Choose whether to download one dataset or skip this notebook
- Allow the user to select a target dataset for download
- Present approximate dataset sizes to help guide selection
- Prevent unsupported OpenImages downloads because of its extreme size
- Store the selected execution state in `RUN_DOWNLOAD` and `TARGET_NAME`
- Pause notebook execution until the user enters a valid selection
- Display a short selection summary before continuing execution

---

In [ ]:
# ============================================
# Step 2: Download Mode and Target Selection
# ============================================

# -------------------------------------------------
# Available dataset options
# -------------------------------------------------
TARGET_OPTIONS = {
    "1": ("imagenet", "SMALL (~331 MB)"),
    "2": ("coco", "MEDIUM (~1.5 GB)"),
    "3": ("diffusiondb", "MEDIUM (~1.8 GB)"),
    "4": ("sdxl", "VERY LARGE (~3.7 GB)"),
    "5": ("midjourney", "VERY LARGE (~3.8 GB)"),
    "6": ("skip", "Skip Notebook 01"),
}

print("Select dataset download option:\n")
print("1. ImageNet (SMALL ~331 MB)")
print("2. MS COCO 2017 (MEDIUM ~1.5 GB)")
print("3. DiffusionDB (MEDIUM ~1.8 GB)")
print("4. SDXL (VERY LARGE ~3.7 GB)")
print("5. MidJourney (VERY LARGE ~3.8 GB)")
print("6. Skip Notebook 01")

choice = input("\nEnter selection number: ").strip()

if choice not in TARGET_OPTIONS:
    raise ValueError(f"Invalid selection: {choice}")

TARGET_NAME, dataset_size = TARGET_OPTIONS[choice]

if TARGET_NAME == "skip":
    RUN_DOWNLOAD = False
    TARGET_NAME = None
    print("\nNotebook 01 skipped.")
    print("Proceed to 02_Preprocess_Images.ipynb.")

else:
    RUN_DOWNLOAD = True

    print("\nSelection Summary:")
    print(f"RUN_DOWNLOAD : {RUN_DOWNLOAD}")
    print(f"TARGET_NAME  : {TARGET_NAME}")
    print(f"Dataset Size : {dataset_size}")



Select dataset download option:

1. ImageNet (SMALL ~331 MB)
2. MS COCO 2017 (MEDIUM ~1.5 GB)
3. DiffusionDB (MEDIUM ~1.8 GB)
4. SDXL (VERY LARGE ~3.7 GB)
5. MidJourney (VERY LARGE ~3.8 GB)
6. Skip Notebook 01

Enter selection number: 1

Selection Summary:
RUN_DOWNLOAD : True
TARGET_NAME  : imagenet
Dataset Size : SMALL (~331 MB)


### 🔷 Step 3 — Load Target Module

- Define dataset configuration mapping for all supported datasets
- Validate the selected dataset (`TARGET_NAME`)
- Extract dataset-specific parameters (name, label, code)
- Resolve the corresponding dataset module file path
- Dynamically import the selected dataset module
- Skip module loading if no valid dataset is selected
- Optionally display dataset configuration when `VERBOSE=True`

---


In [ ]:
# ============================================
# Step 3: Load Target Module
# ============================================

import importlib.util
import os

# -------------------------------------------------
# Dataset configuration mapping
# -------------------------------------------------
# Defines metadata and module mapping for each dataset
TARGET_SPECS = {
    "diffusiondb": {
        "display_name": "DiffusionDB",
        "module_name": "diffusiondb_target",
        "source_dataset": "DiffusionDB",
        "class_label": "ai",
        "dataset_code": "diff",
    },
    "sdxl": {
        "display_name": "SDXL Generated (10K)",
        "module_name": "sdxl_target",
        "source_dataset": "SDXL_Generated_10K",
        "class_label": "ai",
        "dataset_code": "sdxl",
    },
    "imagenet": {
        "display_name": "ImageNet (256)",
        "module_name": "imagenet_target",
        "source_dataset": "ImageNet_1K_256",
        "class_label": "rl",
        "dataset_code": "imgn",
    },
    "coco": {
        "display_name": "MS COCO 2017",
        "module_name": "coco_target",
        "source_dataset": "MS_COCO_2017",
        "class_label": "rl",
        "dataset_code": "coco",
    },
    "midjourney": {
        "display_name": "Midjourney",
        "module_name": "midjourney_target",
        "source_dataset": "Midjourney",
        "class_label": "ai",
        "dataset_code": "midj",
    },
    "openimages": {
        "display_name": "OpenImages",
        "module_name": "openimages_target",
        "source_dataset": "OpenImages",
        "class_label": "rl",
        "dataset_code": "open",
    },
}

# -------------------------------------------------
# Skip module loading if no valid download selected
# -------------------------------------------------
if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping module load (no valid dataset download selected).")

    # Reset all target-related variables
    TARGET_SPEC = None
    DISPLAY_NAME = None
    MODULE_NAME = None
    SOURCE_DATASET = None
    CLASS_LABEL = None
    DATASET_CODE = None
    target_module = None

# -------------------------------------------------
# Load selected dataset module
# -------------------------------------------------
else:
    # Validate selected dataset
    if TARGET_NAME not in TARGET_SPECS:
        raise ValueError(f"Unsupported TARGET_NAME: {TARGET_NAME}")

    # Retrieve dataset configuration
    TARGET_SPEC = TARGET_SPECS[TARGET_NAME]

    DISPLAY_NAME   = TARGET_SPEC["display_name"]
    MODULE_NAME    = TARGET_SPEC["module_name"]
    SOURCE_DATASET = TARGET_SPEC["source_dataset"]
    CLASS_LABEL    = TARGET_SPEC["class_label"]
    DATASET_CODE   = TARGET_SPEC["dataset_code"]

    # -------------------------------------------------
    # Resolve module file path
    # -------------------------------------------------
    module_file = os.path.join(PROJECT_SRC_DIR, "datasets", f"{MODULE_NAME}.py")
    if not os.path.isfile(module_file):
        raise FileNotFoundError(f"Target module file not found: {module_file}")

    # -------------------------------------------------
    # Dynamically load target dataset module
    # -------------------------------------------------
    spec = importlib.util.spec_from_file_location(MODULE_NAME, module_file)
    target_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(target_module)

    print("Target module loaded successfully.")

    # -------------------------------------------------
    # Optional verbose display
    # -------------------------------------------------
    if VERBOSE:
        print()
        print(f"DISPLAY_NAME   : {DISPLAY_NAME}")
        print(f"MODULE_NAME    : {MODULE_NAME}")
        print(f"SOURCE_DATASET : {SOURCE_DATASET}")
        print(f"CLASS_LABEL    : {CLASS_LABEL}")
        print(f"DATASET_CODE   : {DATASET_CODE}")



Target module loaded successfully.

DISPLAY_NAME   : ImageNet (256)
MODULE_NAME    : imagenet_target
SOURCE_DATASET : ImageNet_1K_256
CLASS_LABEL    : rl
DATASET_CODE   : imgn


### 🔷 Step 4 — Common Path Configuration

- Define dataset-specific paths for:
  - raw image directory
  - image output directory
  - raw metadata CSV
  - source hash JSON
  - global hash JSON
- Create required directories if they do not exist
- Skip configuration if no valid dataset is selected
- Optionally display resolved paths when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 4: Common Path Configuration
# ============================================

# -------------------------------------------------
# Skip configuration if no valid dataset selected
# -------------------------------------------------
if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping common configuration (no valid dataset download selected).")

    # Reset all path variables
    DATASET_RAW_DIR = None
    IMAGE_OUTPUT_DIR = None
    RAW_METADATA_CSV = None
    SOURCE_HASHES_JSON = None
    GLOBAL_HASHES_JSON = None

# -------------------------------------------------
# Build common paths and ensure directories exist
# -------------------------------------------------
else:
    # Root directory for selected dataset
    DATASET_RAW_DIR = os.path.join(cfg.RAW_DATA_DIR, SOURCE_DATASET)

    # Directory for downloaded image files
    IMAGE_OUTPUT_DIR = os.path.join(DATASET_RAW_DIR, "images")

    # CSV file for raw metadata (per dataset)
    RAW_METADATA_CSV = os.path.join(
        cfg.ORIGINAL_METADATA_DIR,
        f"{DATASET_CODE}_raw_metadata.csv"
    )

    # JSON file for dataset-specific image hashes
    SOURCE_HASHES_JSON = os.path.join(
        cfg.HASH_METADATA_DIR,
        f"{DATASET_CODE}_source_hashes.json"
    )

    # JSON file for global hash tracking across datasets
    GLOBAL_HASHES_JSON = os.path.join(
        cfg.HASH_METADATA_DIR,
        "global_hashes.json"
    )

    # -------------------------------------------------
    # Ensure required directories exist
    # -------------------------------------------------
    os.makedirs(DATASET_RAW_DIR, exist_ok=True)
    os.makedirs(IMAGE_OUTPUT_DIR, exist_ok=True)
    os.makedirs(cfg.ORIGINAL_METADATA_DIR, exist_ok=True)
    os.makedirs(cfg.HASH_METADATA_DIR, exist_ok=True)

    print("Common configuration complete.\n")

    # -------------------------------------------------
    # Optional verbose display of resolved paths
    # -------------------------------------------------
    if VERBOSE:
        print(f"DATASET_RAW_DIR    : {DATASET_RAW_DIR}")
        print(f"IMAGE_OUTPUT_DIR   : {IMAGE_OUTPUT_DIR}")
        print(f"RAW_METADATA_CSV   : {RAW_METADATA_CSV}")
        print(f"SOURCE_HASHES_JSON : {SOURCE_HASHES_JSON}")
        print(f"GLOBAL_HASHES_JSON : {GLOBAL_HASHES_JSON}")



Common configuration complete.

DATASET_RAW_DIR    : /content/dip-ai-image-detection/data/raw/ImageNet_1K_256
IMAGE_OUTPUT_DIR   : /content/dip-ai-image-detection/data/raw/ImageNet_1K_256/images
RAW_METADATA_CSV   : /content/dip-ai-image-detection/metadata/original/imgn_raw_metadata.csv
SOURCE_HASHES_JSON : /content/dip-ai-image-detection/metadata/hashes/imgn_source_hashes.json
GLOBAL_HASHES_JSON : /content/dip-ai-image-detection/metadata/hashes/global_hashes.json


### 🔷 Step 5 — Reset Current Source State for Rebuild

- Optionally reset the selected dataset source before rebuilding it
- Remove the current source hashes from the global hash record
- Delete the source-specific hash JSON file
- Delete the source raw metadata CSV
- Delete and recreate the selected source image directory
- Skip reset when no valid dataset is selected or `RESET_SOURCE_FOR_REBUILD=False`

⚠️ This step deletes saved state for the selected dataset source when enabled.

---

In [ ]:
# ============================================
# Step 5: Reset Current Source State for Rebuild
# ============================================

RESET_SOURCE_FOR_REBUILD = True  # Set True only when you want to rebuild this source

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping source reset (no valid dataset selected).")

elif not RESET_SOURCE_FOR_REBUILD:
    print("Source reset skipped.")

else:
    import json
    import shutil

    print(f"Resetting saved state for current source: {SOURCE_DATASET}")

    # ----------------------------------------
    # Load current source hashes directly from file
    # ----------------------------------------
    if os.path.exists(SOURCE_HASHES_JSON):
        with open(SOURCE_HASHES_JSON, "r") as f:
            source_data = json.load(f)

        if isinstance(source_data, dict):
            source_hashes_reset = set(source_data.values())
        elif isinstance(source_data, list):
            source_hashes_reset = set(source_data)
        else:
            raise ValueError(f"Unsupported JSON structure in {SOURCE_HASHES_JSON}")
    else:
        source_hashes_reset = set()

    # ----------------------------------------
    # Load global hashes directly from file
    # ----------------------------------------
    if os.path.exists(GLOBAL_HASHES_JSON):
        with open(GLOBAL_HASHES_JSON, "r") as f:
            global_data = json.load(f)

        if isinstance(global_data, dict):
            global_hashes_reset = set(global_data.values())
        elif isinstance(global_data, list):
            global_hashes_reset = set(global_data)
        else:
            raise ValueError(f"Unsupported JSON structure in {GLOBAL_HASHES_JSON}")
    else:
        global_hashes_reset = set()

    original_source_count = len(source_hashes_reset)
    original_global_count = len(global_hashes_reset)

    # ----------------------------------------
    # Remove current source hashes from global
    # ----------------------------------------
    overlap = source_hashes_reset & global_hashes_reset
    global_hashes_reset -= overlap

    with open(GLOBAL_HASHES_JSON, "w") as f:
        json.dump(sorted(global_hashes_reset), f, indent=2)

    # ----------------------------------------
    # Delete current source hash file
    # ----------------------------------------
    if os.path.exists(SOURCE_HASHES_JSON):
        os.remove(SOURCE_HASHES_JSON)

    # ----------------------------------------
    # Delete current source metadata CSV
    # ----------------------------------------
    if os.path.exists(RAW_METADATA_CSV):
        os.remove(RAW_METADATA_CSV)

    # ----------------------------------------
    # Delete current source image directory
    # ----------------------------------------
    if os.path.exists(DATASET_RAW_DIR):
        shutil.rmtree(DATASET_RAW_DIR)

    # ----------------------------------------
    # Recreate required directories
    # ----------------------------------------
    os.makedirs(DATASET_RAW_DIR, exist_ok=True)
    os.makedirs(IMAGE_OUTPUT_DIR, exist_ok=True)
    os.makedirs(ORIGINAL_METADATA_DIR, exist_ok=True)
    os.makedirs(HASH_METADATA_DIR, exist_ok=True)

    print(f"Original source hash count : {original_source_count}")
    print(f"Original global hash count : {original_global_count}")
    print(f"Removed from global        : {len(overlap)}")
    print(f"New global hash count      : {len(global_hashes_reset)}")
    print("Source hash JSON deleted.")
    print("Source raw metadata CSV deleted.")
    print("Source image directory deleted and recreated.")
    print("Current source reset complete.")



Resetting saved state for current source: ImageNet_1K_256
Original source hash count : 3000
Original global hash count : 18000
Removed from global        : 3000
New global hash count      : 15000
Source hash JSON deleted.
Source raw metadata CSV deleted.
Source image directory deleted and recreated.
Current source reset complete.


### 🔷 Step 6 — Raw Metadata CSV Setup

- Define the raw metadata CSV column schema
- Initialize an empty metadata row container for optional use
- Create or overwrite the raw metadata CSV file
- Write the header row to the CSV file
- Display a warning if the file already exists and will be overwritten
- Skip setup if no valid dataset is selected
- Optionally display the metadata column structure when `VERBOSE=True`

---



In [ ]:
# ============================================
# Step 6: Raw Metadata CSV Setup
# ============================================

import csv

# -------------------------------------------------
# Skip setup if no valid dataset selected
# -------------------------------------------------
if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping raw metadata CSV setup (no valid dataset download selected).")
    RAW_METADATA_COLUMNS = None
    demo_metadata_rows = []

# -------------------------------------------------
# Initialize raw metadata CSV structure
# -------------------------------------------------
else:
    # Define column schema for raw metadata
    RAW_METADATA_COLUMNS = [
        "filename",
        "label",
        "dataset_code",
        "source_name",
        "source_id",
        "source_ref",
        "original_width",
        "original_height",
        "saved_path",
        "sha256",
        "batch_id",
    ]

    # Placeholder for optional demonstration rows
    demo_metadata_rows = []

    # -------------------------------------------------
    # Inform user if file will be created or overwritten
    # -------------------------------------------------
    if VERBOSE:
        if os.path.exists(RAW_METADATA_CSV):
            print("WARNING: Raw metadata CSV already exists and will be overwritten:")
            print(f"  {RAW_METADATA_CSV}")
        else:
            print("Creating new raw metadata CSV:")
            print(f"  {RAW_METADATA_CSV}")

    # -------------------------------------------------
    # Create CSV file and write header row
    # -------------------------------------------------
    with open(RAW_METADATA_CSV, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(RAW_METADATA_COLUMNS)

    print("\nRaw metadata CSV initialized.")

    # -------------------------------------------------
    # Optional verbose display of column schema
    # -------------------------------------------------
    if VERBOSE:
        print("\nMetadata columns:")
        for col in RAW_METADATA_COLUMNS:
            print(f"  - {col}")



Creating new raw metadata CSV:
  /content/dip-ai-image-detection/metadata/original/imgn_raw_metadata.csv

Raw metadata CSV initialized.

Metadata columns:
  - filename
  - label
  - dataset_code
  - source_name
  - source_id
  - source_ref
  - original_width
  - original_height
  - saved_path
  - sha256
  - batch_id


### 🔷 Step 7 — Hash Management and Utilities

- Define utility functions to load and save hash sets from JSON files
- Support both dictionary (source) and list (global) JSON formats
- Load existing source and global hash sets if files are present
- Initialize empty hash sets when files do not exist
- Provide no-op utilities when no valid dataset is selected
- Display hash file status (reuse or creation) when `VERBOSE=True`
- Display counts of loaded source and global hashes

---

In [ ]:
# ============================================
# Step 7: Hash Management and Utilities
# ============================================

import json

# -------------------------------------------------
# Skip hash setup if no valid dataset selected
# -------------------------------------------------
if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping hash utilities setup (no valid dataset download selected).")

    # Initialize empty hash sets
    source_hashes = set()
    global_hashes = set()

    # Define no-op utility functions
    def load_hash_set(json_path):
        return set()

    def save_hash_set(hash_set, json_path):
        return

# -------------------------------------------------
# Define hash utility functions and load existing data
# -------------------------------------------------
else:
    # Load hash set from JSON file (supports dict or list formats)
    def load_hash_set(json_path):
        if os.path.exists(json_path):
            with open(json_path, "r") as f:
                data = json.load(f)

            if isinstance(data, dict):
                return set(data.values())   # source hash JSON: filename -> hash
            elif isinstance(data, list):
                return set(data)            # global hash JSON: [hash, hash, ...]
            else:
                raise ValueError(f"Unsupported JSON structure in {json_path}")

        return set()

    # Save hash set to JSON file (sorted for consistency)
    def save_hash_set(hash_set, json_path):
        with open(json_path, "w") as f:
            json.dump(sorted(hash_set), f, indent=2)

    # -------------------------------------------------
    # Load existing source and global hashes
    # -------------------------------------------------
    source_hashes = load_hash_set(SOURCE_HASHES_JSON)
    global_hashes = load_hash_set(GLOBAL_HASHES_JSON)

    # -------------------------------------------------
    # Inform user about hash file reuse or creation
    # -------------------------------------------------
    if VERBOSE:
        if os.path.exists(SOURCE_HASHES_JSON):
            print("Source hash file already exists and will be reused:")
            print(f"  {SOURCE_HASHES_JSON}")
        else:
            print("Source hash file will be created:")
            print(f"  {SOURCE_HASHES_JSON}")

        if os.path.exists(GLOBAL_HASHES_JSON):
            print("Global hash file already exists and will be reused:")
            print(f"  {GLOBAL_HASHES_JSON}")
        else:
            print("Global hash file will be created:")
            print(f"  {GLOBAL_HASHES_JSON}")

    # -------------------------------------------------
    # Display loaded hash counts
    # -------------------------------------------------
    print("\nLoaded hash counts:")
    print(f"  source_hashes : {len(source_hashes)}")
    print(f"  global_hashes : {len(global_hashes)}")



Source hash file will be created:
  /content/dip-ai-image-detection/metadata/hashes/imgn_source_hashes.json
Global hash file already exists and will be reused:
  /content/dip-ai-image-detection/metadata/hashes/global_hashes.json

Loaded hash counts:
  source_hashes : 0
  global_hashes : 15000


### 🔷 Step 8 — Filename and Counting Utilities

- Define utility functions to count saved images in the output directory
- Determine the next available image index based on existing files
- Generate standardized filenames using dataset code and zero-padded indexing
- Initialize no-op utilities when no valid dataset is selected
- Display current image count and next index when `VERBOSE=True`
- Display an example of the next generated filename

---

In [ ]:
# ============================================
# Step 8: Filename and Counting Utilities
# ============================================

# -------------------------------------------------
# Skip utility setup if no valid dataset selected
# -------------------------------------------------
if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping filename and counting utilities (no valid dataset download selected).")

    # Define no-op utility functions
    def count_saved_images():
        return 0

    def get_next_image_index():
        return 1

    def make_filename(index):
        return None

# -------------------------------------------------
# Define filename and counting utilities
# -------------------------------------------------
else:
    # Count number of saved PNG images in output directory
    def count_saved_images():
        if not os.path.exists(IMAGE_OUTPUT_DIR):
            return 0
        return len([
            f for f in os.listdir(IMAGE_OUTPUT_DIR)
            if f.lower().endswith(".png")
        ])

    # Determine next available image index (1-based)
    def get_next_image_index():
        return count_saved_images() + 1

    # Generate standardized filename using dataset code and index
    def make_filename(index):
        return f"{DATASET_CODE}_{index:06d}.png"

    # -------------------------------------------------
    # Compute current state for display
    # -------------------------------------------------
    current_count = count_saved_images()
    next_index = get_next_image_index()
    sample_filename = make_filename(next_index)

    # -------------------------------------------------
    # Optional verbose display
    # -------------------------------------------------
    if VERBOSE:
        print(f"\nCurrent saved image count : {current_count}")
        print(f"Next image index          : {next_index}")

    # Display example of next filename
    print(f"Next filename example     : {sample_filename}")




Current saved image count : 0
Next image index          : 1
Next filename example     : imgn_000001.png


### 🔷 Step 9 — Image Processing and Hash Generation

- Define utility functions to normalize images to RGB format
- Validate that images meet minimum size requirements
- Compute SHA-256 hashes from normalized image content
- Ensure deterministic hashing by converting images to PNG byte format
- Provide no-op utilities when no valid dataset is selected
- Perform a self-check using a synthetic image when `VERBOSE=True`
- Display validation results including size check and example hash output

---

In [ ]:
# ============================================
# Step 9: Image Processing and Hash Generation
# ============================================

import io
import hashlib
from PIL import Image

# -------------------------------------------------
# Skip utility setup if no valid dataset selected
# -------------------------------------------------
if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping image utilities (no valid dataset download selected).")

    # Define no-op utility functions
    def normalize_to_rgb(image_obj):
        return None

    def meets_min_size(image_obj, min_size=256):
        return False

    def compute_image_sha256(image_obj):
        return None

# -------------------------------------------------
# Define image processing and validation utilities
# -------------------------------------------------
else:
    # Normalize image to RGB format
    def normalize_to_rgb(image_obj):
        """
        Convert input image to RGB PIL Image.
        """
        if image_obj.mode != "RGB":
            image_obj = image_obj.convert("RGB")
        return image_obj

    # Check if image meets minimum dimension requirements
    def meets_min_size(image_obj, min_size=256):
        """
        Return True if both width and height meet minimum size.
        """
        width, height = image_obj.size
        return width >= min_size and height >= min_size

    # Compute SHA-256 hash of normalized image content
    def compute_image_sha256(image_obj):
        """
        Compute SHA-256 hash from normalized PNG byte content.
        """
        image_obj = normalize_to_rgb(image_obj)

        # Convert image to PNG byte stream
        buffer = io.BytesIO()
        image_obj.save(buffer, format="PNG")
        image_bytes = buffer.getvalue()

        # Generate SHA-256 hash from byte content
        return hashlib.sha256(image_bytes).hexdigest()

    print("Image utilities ready.\n")

    # -------------------------------------------------
    # Optional self-check for utility validation
    # -------------------------------------------------
    if VERBOSE:
        # Create synthetic test image
        test_img = Image.new("RGB", (256, 256), color=(0, 0, 0))
        test_hash = compute_image_sha256(test_img)

        # Display validation results
        print(f"Minimum size check passed : {meets_min_size(test_img)}")
        print(f"Example SHA-256 length    : {len(test_hash)}")
        print(f"Example SHA-256 prefix    : {test_hash[:16]}...")



Image utilities ready.

Minimum size check passed : True
Example SHA-256 length    : 64
Example SHA-256 prefix    : 3b93f26267630963...


### 🔷 Step 10 — CSV Append Utility

- Define a utility function to append a single metadata row to the raw metadata CSV
- Skip setup when no valid dataset is selected
- Optionally display the target CSV path when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 10: CSV Append Utility
# ============================================

import csv

# -------------------------------------------------
# Skip utility setup if no valid dataset selected
# -------------------------------------------------
if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping CSV append utility (no valid dataset download selected).")

    # Initialize empty demo container
    demo_metadata_rows = []

    # Define no-op append function
    def append_metadata_row(row_dict):
        return

# -------------------------------------------------
# Define CSV append utility
# -------------------------------------------------
else:
    # Placeholder for optional demonstration rows
    demo_metadata_rows = []

    # Append a single metadata row to the CSV file
    def append_metadata_row(row_dict):
        """
        Append one accepted image metadata row to CSV.
        """
        with open(RAW_METADATA_CSV, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=RAW_METADATA_COLUMNS)
            writer.writerow(row_dict)

    print("CSV append utility ready.")

    # -------------------------------------------------
    # Optional verbose display of target CSV path
    # -------------------------------------------------
    if VERBOSE:
        print(f"Target CSV : {RAW_METADATA_CSV}")


CSV append utility ready.
Target CSV : /content/dip-ai-image-detection/metadata/original/imgn_raw_metadata.csv


### 🔷 Step 11 — Load Source Dataset

- Load the selected dataset using the dataset-specific module
- Handle and surface errors during dataset loading
- Skip loading when no valid dataset is selected
- Optionally display dataset type and current saved image count when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 11: Load Source Dataset
# ============================================

# -------------------------------------------------
# Skip dataset loading if no valid dataset selected
# -------------------------------------------------
if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping source dataset load (no valid dataset download selected).")
    dataset_obj = None

# -------------------------------------------------
# Load dataset using target module
# -------------------------------------------------
else:
    print("Loading source dataset...\n")

    try:
        # Call dataset-specific loader function
        dataset_obj = target_module.load_source_dataset()
    except Exception as e:
        # Surface loading errors clearly
        print("ERROR: Failed to load source dataset.")
        raise e

    print("Source dataset loaded successfully.\n")

    # -------------------------------------------------
    # Optional verbose display of dataset type
    # -------------------------------------------------
    if VERBOSE:
        print(f"Dataset type : {type(dataset_obj)}")

    # -------------------------------------------------
    # Reflect current saved image state
    # -------------------------------------------------
    current_count = count_saved_images()

    if VERBOSE:
        print(f"\nCurrent accepted images : {current_count}")



Loading source dataset...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Source dataset loaded successfully.

Dataset type : <class 'datasets.iterable_dataset.IterableDataset'>

Current accepted images : 0


### 🔷 Step 12 — Preview Candidate Records

- Retrieve a small batch of candidate records from the dataset module
- Handle and surface errors during record retrieval
- Report when no candidate records are returned
- Optionally display record details and image properties when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 12: Preview Candidate Records
# ============================================

# -------------------------------------------------
# Skip preview if no valid dataset selected
# -------------------------------------------------
if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping candidate preview (no valid dataset download selected).")

# -------------------------------------------------
# Retrieve and inspect sample candidate records
# -------------------------------------------------
else:
    print("Retrieving candidate records...\n")

    try:
        # Request a small batch of candidate records from dataset module
        preview_records = target_module.get_candidate_records(dataset_obj, batch_size=3)
    except Exception as e:
        # Surface retrieval errors clearly
        print("ERROR: Failed to retrieve candidate records.")
        raise e

    # -------------------------------------------------
    # Handle empty result set
    # -------------------------------------------------
    if not preview_records:
        print("No candidate records returned.")
    else:
        print(f"Retrieved {len(preview_records)} candidate records.")

        # -------------------------------------------------
        # Optional verbose inspection of candidate records
        # -------------------------------------------------
        if VERBOSE:
            print()

            for i, record in enumerate(preview_records, start=1):
                print(f"Record {i}:")
                print(f"  source_id  : {record.get('source_id')}")
                print(f"  source_ref : {record.get('source_ref')}")

                # Inspect associated image object if present
                img = record.get("image_obj")
                if img is not None:
                    try:
                        print(f"  image size : {img.size}")
                        print(f"  image mode : {img.mode}")
                    except Exception:
                        print("  image info : <unavailable>")
                else:
                    print("  image_obj  : None")

                print()



Retrieving candidate records...

Retrieved 3 candidate records.

Record 1:
  source_id  : imagenet_0
  source_ref : huggingface://benjamin-paine/imagenet-1k-256x256/train
  image size : (256, 256)
  image mode : RGB

Record 2:
  source_id  : imagenet_1
  source_ref : huggingface://benjamin-paine/imagenet-1k-256x256/train
  image size : (256, 256)
  image mode : RGB

Record 3:
  source_id  : imagenet_2
  source_ref : huggingface://benjamin-paine/imagenet-1k-256x256/train
  image size : (256, 256)
  image mode : RGB



### 🔷 Step 13 — Process One Candidate Record

- Process a single candidate record through the full acceptance pipeline
- Normalize image, apply size filtering, and compute SHA-256 hash
- Perform deduplication against source and global hash sets
- Save accepted images and update hash tracking
- Append metadata for accepted records to the CSV file
- Return acceptance status (`True` or `False`) for each record
- Skip processing when no valid dataset is selected

---


In [ ]:
# ============================================
# Step 13: Process One Candidate Record
# ============================================

# -------------------------------------------------
# Skip processing if no valid dataset selected
# -------------------------------------------------
if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping candidate record processing (no valid dataset download selected).")

    # Define no-op processing function
    def process_candidate_record(record, index):
        return False

# -------------------------------------------------
# Define candidate record processing pipeline
# -------------------------------------------------
else:
    def process_candidate_record(record, index):
        """
        Process a single candidate record.
        Returns True if accepted, False otherwise.
        """

        try:
            # -------------------------------------------------
            # Extract fields from candidate record
            # -------------------------------------------------
            source_id = record.get("source_id")
            source_ref = record.get("source_ref")
            image_obj = record.get("image_obj")

            # Reject if no image present
            if image_obj is None:
                return False

            # -------------------------------------------------
            # Normalize image format
            # -------------------------------------------------
            image_obj = normalize_to_rgb(image_obj)

            # -------------------------------------------------
            # Apply minimum size filter
            # -------------------------------------------------
            if not meets_min_size(image_obj):
                return False

            # -------------------------------------------------
            # Compute image hash
            # -------------------------------------------------
            sha256 = compute_image_sha256(image_obj)

            # -------------------------------------------------
            # Deduplicate against source and global hashes
            # -------------------------------------------------
            if sha256 in source_hashes:
                return False

            if sha256 in global_hashes:
                return False

            # -------------------------------------------------
            # Generate filename for accepted image
            # -------------------------------------------------
            filename = make_filename(index)

            # -------------------------------------------------
            # Save accepted image to disk
            # -------------------------------------------------
            saved_path = os.path.join(IMAGE_OUTPUT_DIR, filename)
            image_obj.save(saved_path, format="PNG")

            # -------------------------------------------------
            # Update hash tracking sets
            # -------------------------------------------------
            source_hashes.add(sha256)
            global_hashes.add(sha256)

            # -------------------------------------------------
            # Build metadata row for accepted image
            # -------------------------------------------------
            row = {
                "filename": filename,
                "label": CLASS_LABEL,
                "dataset_code": DATASET_CODE,
                "source_name": SOURCE_DATASET,
                "source_id": source_id,
                "source_ref": source_ref,
                "original_width": image_obj.size[0],
                "original_height": image_obj.size[1],
                "saved_path": saved_path,
                "sha256": sha256,
                "batch_id": None,
            }

            # Append metadata to CSV
            append_metadata_row(row)

            return True

        except Exception:
            # Gracefully handle processing errors
            return False

    print("Candidate record processing function ready.")



Candidate record processing function ready.


### 🔷 Step 14 — Build Dataset for Selected Source

- Execute the full dataset build pipeline for the selected source
- Retrieve candidate records in batches using module-supported or iterator-based methods
- Process each record through the acceptance pipeline (Step 13)
- Track accepted, rejected, and processed counts
- Persist hash state after each batch to maintain deduplication
- Stop when target image count is reached or no more candidates are available
- Optionally display batch progress and statistics when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 14: Build Dataset for Selected Source
# ============================================

from tqdm.notebook import tqdm
import inspect
import itertools

# -------------------------------------------------
# Skip dataset build if no valid dataset selected
# -------------------------------------------------
if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping dataset build (no valid dataset download selected).")

# -------------------------------------------------
# Define dataset build pipeline
# -------------------------------------------------
else:
    def build_dataset_for_selected_source(target_count=TARGET_COUNT, batch_size=BATCH_SIZE, max_batches=None):
        """
        Build dataset for the selected source until target_count accepted images are reached
        or candidate records are exhausted.
        Supports:
          - target modules with start_index support
          - iterable datasets via notebook-side iteration
        """

        # -------------------------------------------------
        # Initialize counters and state
        # -------------------------------------------------
        accepted_count = 0
        rejected_count = 0
        processed_count = 0
        batch_id = 0
        next_index = get_next_image_index()

        print("Starting dataset build...\n")
        print(f"Target source     : {SOURCE_DATASET}")
        print(f"Target count      : {target_count}")
        print(f"Batch size        : {batch_size}")

        # -------------------------------------------------
        # Determine batching strategy based on module support
        # -------------------------------------------------
        sig = inspect.signature(target_module.get_candidate_records)
        supports_start_index = "start_index" in sig.parameters

        if supports_start_index:
            batching_mode = "module start_index"
            source_iter = None
            candidate_offset = 0
        else:
            batching_mode = "notebook iterator"
            source_iter = iter(dataset_obj)
            candidate_offset = None

        if VERBOSE:
            print(f"Batching mode     : {batching_mode}\n")

        # Progress bar (only visible in verbose mode)
        pbar = tqdm(total=target_count, desc="Accepted Images", unit="img", disable=not VERBOSE)

        # -------------------------------------------------
        # Main dataset build loop
        # -------------------------------------------------
        while accepted_count < target_count:
            batch_id += 1

            # Optional batch limit for debugging/testing
            if max_batches is not None and batch_id > max_batches:
                print("Reached max_batches limit.")
                break

            try:
                # -----------------------------------------
                # Retrieve candidate records (two modes)
                # -----------------------------------------
                if supports_start_index:
                    candidate_records = target_module.get_candidate_records(
                        dataset_obj,
                        start_index=candidate_offset,
                        batch_size=batch_size
                    )
                    candidate_offset += batch_size
                else:
                    raw_batch = list(itertools.islice(source_iter, batch_size))
                    if not raw_batch:
                        candidate_records = []
                    else:
                        candidate_records = target_module.get_candidate_records(
                            raw_batch,
                            batch_size=batch_size
                        )

            except Exception as e:
                print(f"ERROR: Failed to retrieve candidate records for batch {batch_id}.")
                raise e

            # Stop if no more data
            if not candidate_records:
                print("No more candidate records returned. Stopping build.")
                break

            batch_accepted = 0
            batch_rejected = 0

            # -----------------------------------------
            # Process each candidate record
            # -----------------------------------------
            for record in candidate_records:
                processed_count += 1

                accepted = process_candidate_record(record, next_index)

                if accepted:
                    accepted_count += 1
                    batch_accepted += 1
                    next_index += 1
                    pbar.update(1)
                else:
                    rejected_count += 1
                    batch_rejected += 1

                # Stop once target is reached
                if accepted_count >= target_count:
                    break

            # -----------------------------------------
            # Persist hash state after each batch
            # -----------------------------------------
            save_hash_set(source_hashes, SOURCE_HASHES_JSON)
            save_hash_set(global_hashes, GLOBAL_HASHES_JSON)

            # Format offset display depending on mode
            if supports_start_index:
                offset_text = f"{candidate_offset:5d}"
            else:
                offset_text = "iter "

            # Optional verbose batch summary
            if VERBOSE:
                print(
                    f"Batch {batch_id:03d} | "
                    f"offset: {offset_text} | "
                    f"processed: {len(candidate_records):4d} | "
                    f"accepted: {batch_accepted:4d} | "
                    f"rejected: {batch_rejected:4d} | "
                    f"total accepted: {accepted_count:4d}"
                )

            # Stop if final partial batch
            if len(candidate_records) < batch_size:
                print("Final partial batch reached. Stopping build.")
                break

        pbar.close()

        # -------------------------------------------------
        # Final summary
        # -------------------------------------------------
        print("\nDataset build complete.\n")
        print(f"Total processed : {processed_count}")
        print(f"Total accepted  : {accepted_count}")
        print(f"Total rejected  : {rejected_count}")
        print(f"Images saved to : {IMAGE_OUTPUT_DIR}")
        print(f"Metadata CSV    : {RAW_METADATA_CSV}")

        return {
            "processed_count": processed_count,
            "accepted_count": accepted_count,
            "rejected_count": rejected_count,
            "batches_run": batch_id,
        }

    print("Dataset build function ready.")



Dataset build function ready.


### 🔷 Step 15 — Verify Saved Output

- Verify consistency between saved images and metadata records
- Count saved image files and metadata CSV rows
- Report whether counts match or indicate a mismatch
- Skip verification when no valid dataset is selected

---

In [ ]:
# ============================================
# Step 15: Verify Saved Output
# ============================================

import csv

# -------------------------------------------------
# Skip verification if no valid dataset selected
# -------------------------------------------------
if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping output verification (no valid dataset download selected).")

# -------------------------------------------------
# Define output verification utility
# -------------------------------------------------
else:
    def verify_saved_output():
        print("Verifying output...\n")

        # -------------------------------------------------
        # Count saved image files
        # -------------------------------------------------
        image_count = count_saved_images()

        # -------------------------------------------------
        # Count metadata rows (excluding header)
        # -------------------------------------------------
        if os.path.exists(RAW_METADATA_CSV):
            with open(RAW_METADATA_CSV, "r", newline="") as f:
                reader = csv.reader(f)
                row_count = sum(1 for _ in reader) - 1  # exclude header
        else:
            row_count = 0

        # -------------------------------------------------
        # Display verification results
        # -------------------------------------------------
        print(f"Saved images       : {image_count}")
        print(f"Metadata CSV rows  : {row_count}")

        # -------------------------------------------------
        # Check consistency between images and metadata
        # -------------------------------------------------
        if image_count == row_count:
            print("\nVerification passed: image count matches metadata row count.")
        else:
            print("\nWARNING: image count does not match metadata row count.")

        # Return structured verification results
        return {
            "image_count": image_count,
            "metadata_row_count": row_count,
            "match": (image_count == row_count),
        }

    print("Output verification function ready.")



Output verification function ready.


### 🔷 Step 16 — Run Dataset Build and Verification

- Execute the full dataset build pipeline for the selected source
- Display summary statistics from the build process
- Run output verification to confirm dataset integrity
- Display verification results
- Skip execution when no valid dataset is selected

---

In [ ]:
# ============================================
# Step 16: Run Dataset Build and Verification
# ============================================

# -------------------------------------------------
# Skip execution if no valid dataset selected
# -------------------------------------------------
if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping dataset build and verification (no valid dataset selected).")

# -------------------------------------------------
# Execute dataset build and verification pipeline
# -------------------------------------------------
else:
    print("Running dataset build...\n")

    # -------------------------------------------------
    # Run dataset builder (Step 14)
    # -------------------------------------------------
    results = build_dataset_for_selected_source(
        target_count=TARGET_COUNT,
        batch_size=BATCH_SIZE
    )

    # -------------------------------------------------
    # Display build summary
    # -------------------------------------------------
    print("\nBuild Summary:")
    print(f"Processed : {results['processed_count']}")
    print(f"Accepted  : {results['accepted_count']}")
    print(f"Rejected  : {results['rejected_count']}")
    print(f"Batches   : {results['batches_run']}")

    print("\nRunning verification...\n")

    # -------------------------------------------------
    # Run output verification (Step 15)
    # -------------------------------------------------
    verification = verify_saved_output()

    # -------------------------------------------------
    # Display verification summary
    # -------------------------------------------------
    print("\nVerification Summary:")
    for key, value in verification.items():
        print(f"{key} : {value}")



Running dataset build...

Starting dataset build...

Target source     : ImageNet_1K_256
Target count      : 3000
Batch size        : 200
Batching mode     : notebook iterator



Accepted Images:   0%|          | 0/3000 [00:00<?, ?img/s]

Batch 001 | offset: iter  | processed:  200 | accepted:  200 | rejected:    0 | total accepted:  200
Batch 002 | offset: iter  | processed:  200 | accepted:  200 | rejected:    0 | total accepted:  400
Batch 003 | offset: iter  | processed:  200 | accepted:  200 | rejected:    0 | total accepted:  600
Batch 004 | offset: iter  | processed:  200 | accepted:  200 | rejected:    0 | total accepted:  800
Batch 005 | offset: iter  | processed:  200 | accepted:  200 | rejected:    0 | total accepted: 1000
Batch 006 | offset: iter  | processed:  200 | accepted:  200 | rejected:    0 | total accepted: 1200
Batch 007 | offset: iter  | processed:  200 | accepted:  200 | rejected:    0 | total accepted: 1400
Batch 008 | offset: iter  | processed:  200 | accepted:  200 | rejected:    0 | total accepted: 1600
Batch 009 | offset: iter  | processed:  200 | accepted:  200 | rejected:    0 | total accepted: 1800
Batch 010 | offset: iter  | processed:  200 | accepted:  200 | rejected:    0 | total accep

### 🔷 Step 17 — Final Summary Report

- Display final dataset summary including counts and output locations
- Report source and global hash totals
- Compute and display acceptance rate when available
- Indicate completion of Notebook 01 execution
- Skip summary when no dataset was built

---

In [ ]:
# ============================================
# Step 17: Final Summary Report
# ============================================

import time

# -------------------------------------------------
# Skip summary if no dataset was built
# -------------------------------------------------
if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("No dataset was built in this session.")

# -------------------------------------------------
# Display final summary report
# -------------------------------------------------
else:
    print("========== FINAL SUMMARY ==========\n")

    # -------------------------------------------------
    # Dataset identification
    # -------------------------------------------------
    print(f"Source Dataset : {SOURCE_DATASET}")
    print(f"Dataset Code   : {DATASET_CODE}")

    # ----------------------------------------
    # Output counts and locations
    # ----------------------------------------
    image_count = count_saved_images()

    print(f"\nSaved Images   : {image_count}")
    print(f"Metadata CSV   : {RAW_METADATA_CSV}")
    print(f"Image Dir      : {IMAGE_OUTPUT_DIR}")

    # Hash tracking summary
    print(f"Source Hashes  : {len(source_hashes)}")
    print(f"Global Hashes  : {len(global_hashes)}")

    # ----------------------------------------
    # Efficiency metrics (if build results available)
    # ----------------------------------------
    try:
        processed = results["processed_count"]
        accepted  = results["accepted_count"]

        # Compute acceptance rate
        accept_rate = accepted / processed if processed > 0 else 0

        print("\n--- Performance ---")
        print(f"Total Processed : {processed}")
        print(f"Total Accepted  : {accepted}")
        print(f"Acceptance Rate : {accept_rate:.4f}")

    except Exception:
        # Handle missing results gracefully
        print("\n(No performance stats available)")

    # ----------------------------------------
    # Completion message
    # ----------------------------------------
    print("===================================")
    print("\nNotebook 01 execution complete.")



========== FINAL SUMMARY ==========

Source Dataset : ImageNet_1K_256
Dataset Code   : imgn

Saved Images   : 3000
Metadata CSV   : /content/dip-ai-image-detection/metadata/original/imgn_raw_metadata.csv
Image Dir      : /content/dip-ai-image-detection/data/raw/ImageNet_1K_256/images
Source Hashes  : 3000
Global Hashes  : 18000

--- Performance ---
Total Processed : 3000
Total Accepted  : 3000
Acceptance Rate : 1.0000

Notebook 01 execution complete.
